## Preamble and sampling


In [1]:
import nltk
import polars as pl

from collections import Counter
from datasets import load_dataset
from nltk.tokenize import word_tokenize

/home/ubuntu/miniconda/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
load_dataset("roneneldan/TinyStories", split="train").to_parquet("tinystories.parquet")
load_dataset("SimpleStories/SimpleStories", split="train").to_parquet("simplestories.parquet")

Creating parquet from Arrow format: 100%|██████████| 32/32 [00:57<00:00,  1.80s/ba]


3142783327

In [84]:
sample_size = 10000

In [85]:
tinystories_df = pl.scan_parquet("tinystories.parquet")
# tinystories_df = pl.scan_parquet("tinystories.parquet").head(50000).collect()

In [86]:
simplestories_df = pl.scan_parquet("simplestories.parquet")
# simplestories_df = pl.scan_parquet("simplestories.parquet").head(50000).collect()

## Length comparison

In [87]:
tinystories_df = tinystories_df.with_columns(
    pl.col("text").str.split(" ").list.len().alias("word_count")
)

In [88]:
%%time
"""
flesch-kincaid approx function in pure polars
"""
tinystories_df = tinystories_df.with_columns([
    # Count words
    pl.col("text").str.split(" ").list.len().alias("word_count"),
    # Count sentences (split on . ! ?)
    pl.col("text").str.count_matches(r"[.!?]+").alias("sentence_count"),
    # Approximate syllables: count vowel groups per word
    pl.col("text").str.count_matches(r"[aeiouAEIOU]+").alias("syllable_count"),
]).with_columns(
    (
        0.39 * (pl.col("word_count") / pl.col("sentence_count").clip(lower_bound=1))
        + 11.8 * (pl.col("syllable_count") / pl.col("word_count").clip(lower_bound=1))
        - 15.59
    ).alias("flesch_kincaid_score")
)

simplestories_df = simplestories_df.with_columns([
    # Count words
    pl.col("story").str.split(" ").list.len().alias("word_count"),
    # Count sentences (split on . ! ?)
    pl.col("story").str.count_matches(r"[.!?]+").alias("sentence_count"),
    # Approximate syllables: count vowel groups per word
    pl.col("story").str.count_matches(r"[aeiouAEIOU]+").alias("syllable_count"),
]).with_columns(
    (
        0.39 * (pl.col("word_count") / pl.col("sentence_count").clip(lower_bound=1))
        + 11.8 * (pl.col("syllable_count") / pl.col("word_count").clip(lower_bound=1))
        - 15.59
    ).alias("flesch_kincaid_score")
)

CPU times: user 280 μs, sys: 0 ns, total: 280 μs
Wall time: 261 μs


In [89]:
ts_result = tinystories_df.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_score").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_score").std(),
    
)

ss_result = simplestories_df.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_grade").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_grade").std(),
    
)

In [93]:
print("Tiny stories")
ts_result.collect()

Tiny stories


word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
171.832831,77.416249,4.108448,1.513775


In [94]:
print("simple stories")
ss_result.collect()

simple stories


word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
222.87192,102.668153,3.083553,1.235925


In [29]:
simplestories_df.head()

story,topic,theme,style,feature,grammar,persona,initial_word_type,initial_letter,word_count,character_count,num_paragraphs,avg_word_length,avg_sentence_length,flesch_reading_ease,flesch_kincaid_grade,dale_chall_readability_score,num_stories_in_completion,expected_num_stories_in_completion,generation_id,model
str,str,str,str,str,str,str,str,str,i64,i64,i64,f64,f64,f64,f64,f64,i64,i64,str,str
"""Eagerly, a girl named Kim went…","""subterranean worlds""","""Hardship""","""lighthearted""","""symbolism""","""""","""""","""adverb""","""T""",109,570,2,4.34,15.57,74.49,6.3,8.26,11,12,"""7482221ff291ce5936eccc77c700d9…","""gpt-4o-mini-2024-07-18"""
"""Key turned in the old lock. A …","""shape-shifting""","""Failure""","""playful""","""symbolism""","""""","""""","""noun""","""K""",391,1769,6,3.7,10.29,96.28,2.0,7.14,6,6,"""940305166197525600cfc5fcbbb513…","""gpt-4o-mini-2024-07-18"""
"""Rain poured down on the tiny t…","""seasonal changes""","""Friendship""","""humorous""","""irony""","""perfect aspect""","""someone evil""","""noun""","""C""",510,2077,7,3.34,9.81,96.28,2.0,6.09,4,5,"""315219dcf08e93102350aebb412a2c…","""gpt-4o-mini-2024-07-18"""
"""An old tree stood tall in a fo…","""virtual worlds""","""Discovery""","""whimsical""","""circular narrative structure""","""""","""""","""adjective""","""O""",120,567,2,3.84,15.0,83.36,4.9,6.83,11,12,"""1a854e00b6750677b6856a0be0530e…","""gpt-4o-mini-2024-07-18"""
"""Truthfully, the garden was a p…","""gardens""","""Contradiction""","""suspenseful""","""climactic structure""","""""","""""","""adverb""","""T""",466,2148,8,3.77,11.1,87.31,3.4,7.13,4,4,"""5f5524c5a3ac334bdb6c576e531304…","""gpt-4o-mini-2024-07-18"""
